# CT augmentation with MedAugmentX

CT is the modality where naive augmentation does the most damage, because CT
intensities are **not** arbitrary. A Hounsfield unit is a calibrated physical
measurement: water is 0 HU, air is -1000 HU, cortical bone is ~+1000 HU. A
brightness jitter that would be harmless on a photograph turns fat into muscle.

This notebook shows how to augment CT while keeping HU meaningful, and covers
the CT-specific artifact models.
### Before you start

Everything below runs on **synthetic phantoms** built from analytic shapes —
no patient data, no downloads, no network access. They are illustrations, not
validated physical models, so don't read clinical conclusions off them. The
transform strengths are deliberately exaggerated so the effect is visible in a
single figure; they are *not* recommended training policies.

Install what this notebook needs:

```bash
pip install "medaugmentx[notebooks]"
```


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import medaugmentx

print("MedAugmentX", medaugmentx.__version__)


def show(*panels, limits=None, cmap="gray", title=None):
    """Display `(label, volume)` pairs side by side on one fixed grey scale.

    A shared scale matters: normalising each panel independently would hide
    exactly the intensity shifts these transforms are meant to introduce.
    """
    fig, axes = plt.subplots(1, len(panels), figsize=(4.2 * len(panels), 4.4))
    axes = np.atleast_1d(axes)
    planes = []
    for _, volume in panels:
        image = volume.image if hasattr(volume, "image") else volume
        planes.append(image if image.ndim == 2 else image[image.shape[0] // 2])
    lo, hi = limits if limits else (min(p.min() for p in planes), max(p.max() for p in planes))
    for ax, (label, _), plane in zip(axes, panels, planes):
        ax.imshow(plane, cmap=cmap, vmin=lo, vmax=hi, interpolation="nearest")
        ax.set_title(label, fontsize=11)
        ax.set_axis_off()
    if title:
        fig.suptitle(title, fontsize=12)
    fig.tight_layout()
    plt.show()


## 1. A synthetic abdominal slice, in Hounsfield units

Note the value range below — this is calibrated data, not `[0, 1]` pixels.


In [ ]:
from medaugmentx.phantoms import ct_phantom

volume = ct_phantom()
print("shape   ", volume.shape)
print("modality", volume.modality)
print("HU range", (float(volume.image.min()), float(volume.image.max())))

# A soft-tissue window, the way a radiologist would view it.
show(("Synthetic CT (soft-tissue window)", volume), limits=(-250, 350))

## 2. Windowing is the display step — and an augmentation

Radiologists never look at the full HU range at once; they window it. Two
readers may pick slightly different window settings for the same study, so
window variation is a genuine source of appearance variance to train against.

`WindowLevel` perturbs the centre and width. Below, the *same* volume under
three windows — the anatomy is identical, the appearance is not.


In [ ]:
windows = {"lung (-600, 1500)": (-1350, 150),
           "soft tissue (50, 400)": (-150, 250),
           "bone (500, 2000)": (-500, 1500)}

fig, axes = plt.subplots(1, 3, figsize=(13, 4.4))
for ax, (label, (lo, hi)) in zip(axes, windows.items()):
    ax.imshow(volume.image, cmap="gray", vmin=lo, vmax=hi, interpolation="nearest")
    ax.set_title(label, fontsize=11)
    ax.set_axis_off()
fig.suptitle("One volume, three windows", fontsize=12)
fig.tight_layout()
plt.show()

In [ ]:
from medaugmentx.transforms import WindowLevel

# rescale_output=False keeps the result in HU, which is what you want when a
# later transform (or your model) depends on calibrated values.
windowed = WindowLevel(center_shift_frac=0.15, width_scale=(0.8, 1.2),
                       rescale_output=False, seed=7)(volume)

print(f"HU range before: ({volume.image.min():8.1f}, {volume.image.max():8.1f})")
print(f"HU range after : ({windowed.image.min():8.1f}, {windowed.image.max():8.1f})")

show(("Original", volume), ("WindowLevel", windowed), limits=(-250, 350))

## 3. Beam hardening

A polychromatic X-ray beam loses its low-energy photons first as it passes
through dense tissue, so deep structures read darker than they should — cupping
across the body and dark bands between dense objects. `alpha` sets the strength.


In [ ]:
from medaugmentx.transforms import BeamHardening

hardened = BeamHardening(alpha=0.10, seed=7)(volume)
show(("Original", volume), ("BeamHardening(alpha=0.10)", hardened), limits=(-250, 350))

## 4. Metal streak artifact

Hip implants, dental work, and surgical clips throw bright and dark streaks
across the image. Models trained without them tend to fail exactly on the
post-surgical patients you most want to get right.


In [ ]:
from medaugmentx.transforms import MetalStreak

streaked = MetalStreak(intensity=0.16, num_streaks=9, seed=7)(volume)
show(("Original", volume), ("MetalStreak", streaked), limits=(-250, 350))

## 5. Guarding against implausible output

Augmentation strength is easy to overshoot. `VolumeValidator` encodes what a
plausible volume looks like — here, that CT values stay inside a physical HU
range — and `Guard` wraps a transform so violations are caught rather than
silently trained on.

`on_fail="retry"` resamples; `"revert"` returns the input unchanged; `"raise"`
stops. Below, a deliberately absurd transform is caught.


In [ ]:
from medaugmentx import Guard, VolumeValidator
from medaugmentx.transforms import BrightnessContrast

# strict_bounds makes an out-of-range result an error rather than a warning.
validator = VolumeValidator(intensity_bounds=(-1024.0, 3071.0), strict_bounds=True)

print("plausible as-is:", validator.validate(volume).ok)

reckless = BrightnessContrast(brightness=(4000.0, 4000.0), contrast=(1.0, 1.0), seed=1)
report = validator.validate(reckless(volume))
print("after a reckless brightness shift:", report.ok)
for issue in report.errors:
    print("   ", issue)

guarded = Guard(reckless, validator=validator, on_fail="revert")
result = guarded(volume)
print("\nGuard(on_fail='revert') kept the input:", np.array_equal(result.image, volume.image))

## 6. A preset pipeline

`ct_pipeline()` composes the CT defaults. Print it before you trust it.


In [ ]:
from medaugmentx import pipeline_summary
from medaugmentx.presets import ct_pipeline

pipeline = ct_pipeline(seed=0)
print(pipeline_summary(pipeline))

show(("Original", volume), ("ct_pipeline(seed=0)", pipeline(volume)), limits=(-250, 350))

## Where to go next

- **[`docs/API_REFERENCE.md`](../docs/API_REFERENCE.md)** — every public import.
- **[`docs/RESEARCH_GUIDE.md`](../docs/RESEARCH_GUIDE.md)** — replay, worker
  seeds, and how to report augmentation in a paper.
- **[`examples/`](../examples/)** — runnable scripts, including
  `safe_augmentation.py` (validator + `Guard`) and `keypoints_bboxes.py`.
- **The other tutorials** — `01_mri_augmentation.ipynb`,
  `02_ct_augmentation.ipynb`, `03_dbt_augmentation.ipynb`.

Choosing augmentation strength is an empirical question for *your* dataset and
task. Start weak, look at the images, and validate against a held-out set.
